### Test the chroma db working fine or not

In [ ]:
# !pip install chromadb
# !pip install langchain-community langchain-chroma langchain-openai chromadb tqdm
#!pip install langchain-core

In [ ]:
import os
import chromadb

# Step up twice: out of utils, out of src, into the root project directory
db_dir = "../../data/processed/chroma_db"

print(f"Targeting: {db_dir}")
if os.path.exists(db_dir):
    print(f"[FOUND SYSTEM] Path exists. Directory contents: {os.listdir(db_dir)}")
    
    try:
        # Connect using the verified path
        client = chromadb.PersistentClient(path=db_dir)
        collections = client.list_collections()
        
        print(f"\n[SUCCESS] Native client successfully connected in Codespaces!")
        for col in collections:
            print(f" -> Collection Name: '{col.name}' | Total Vectors: {col.count()}")
            
    except Exception as e:
        print(f"\n[ERROR] Found folder, but failed to parse database: {e}")
else:
    print(f"\n[ERROR] Path '{db_dir}' still not found.")
    # Debug trace to show you exactly where the notebook thinks it is
    print("Current absolute location:", os.path.abspath("."))

In [ ]:
import os
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate

# 1. Setup our verified path and embedding model
db_dir = "../../data/processed/chroma_db"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("Initializing Chroma connection...")
db = Chroma(persist_directory=db_dir, embedding_function=embeddings)

# --- TEST 1: RETRIEVAL ---
test_query = "semiconductor supply chain vulnerabilities"
print(f"\n[TEST 1] Running Retrieval for query: '{test_query}'...")

# Fetch top 2 most relevant document chunks
retrieved_docs = db.similarity_search(test_query, k=2)

if retrieved_docs:
    print(f"[RETRIEVAL SUCCESS] Found {len(retrieved_docs)} matching chunks.")
    for i, doc in enumerate(retrieved_docs):
        print(f"\n--- Chunk {i+1} Details ---")
        print(f"Source Document: {doc.metadata.get('file_name', 'Unknown')}")
        print(f"Text Snippet (First 200 chars):\n{doc.page_content[:200]}...")
else:
    print("[RETRIEVAL FAILED] No matching documents found. Check your query terms.")


# --- TEST 2: AUGMENTATION ---
print("\n" + "="*50)
print("[TEST 2] Testing System Augmentation (Prompt Engineering)...")

# Define a standard RAG template
rag_template = """
You are an expert assistant analyzing supply chain data. Use the following pieces of retrieved context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Retrieved Context:
{context}

Question: {question}
Answer:
"""

prompt_template = PromptTemplate(
    template=rag_template, 
    input_variables=["context", "question"]
)

if retrieved_docs:
    # Combine the text from our retrieved chunks into a single context block
    combined_context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # Inject the retrieved context and question into our prompt skeleton
    final_augmented_prompt = prompt_template.format(
        context=combined_context, 
        question=test_query
    )
    
    print("[AUGMENTATION SUCCESS] Prompt engineered successfully!")
    print("\n--- Final Augmented Prompt Ready for LLM Consumption ---")
    print(final_augmented_prompt[:600] + "\n... [Truncated for Display] ...")
else:
    print("[AUGMENTATION SKIPPED] No context docs available to construct the prompt.")

## Create Chroma DB step by step

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
import glob
from llama_parse import LlamaParse
from langchain_core.documents import Document
import nest_asyncio

C:\Users\v2garg\AppData\Local\Temp\ipykernel_2592\962222885.py:5: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


In [2]:
api_key=os.getenv("LLAMA_CLOUD_API_KEY")
api_key

'llx-tWFfQqLUeN2Jb5hI0IOL6qQfl4eLhYKAql8BbT4ZVD5nM85D'

In [4]:
pdf_dir = "C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs/"
pdf_files = glob.glob(os.path.join(pdf_dir, "*.pdf"))
pdf_files

['C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs\\Economic_Security_in_a_Changing_World.pdf',
 'C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs\\vulnerabilities_in_the_semiconductor_supply_chain.pdf']

### Use LlamaParse to load pdf files with images and charts

In [49]:
nest_asyncio.apply()

load_dotenv()

# 2. Initialize the parser using the updated system_prompt parameter
parser = LlamaParse(
    api_key=os.getenv("LLAMA_CLOUD_API_KEY"),
    result_type="markdown",  
    num_workers=4,           
    language="en",
    system_prompt=(
    "You are a strict document parser for a supply chain risk system.\n"
    "1. For narrative text/paragraphs: Copy them EXACTLY as written. No summaries.\n"
    "2. For explicit, printed data tables: Recreate them as Markdown tables.\n"
    "3. For visual graphics (like Figure 1.3 line graphs, charts, maps): DO NOT create a markdown table. Instead, write a detailed 'Visual Analysis Summary' listing the lines/categories, whether they are trending up or down, and approximate values over time."
)
)

# 3. Define path and find the PDFs
pdf_dir = "C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs/"
pdf_files = glob.glob(os.path.join(pdf_dir, "*.pdf"))
print(f"Found {len(pdf_files)} PDF files to process.")

# Try running it again with the fix
langchain_documents = []
for file_path in pdf_files:
    try:
        print(f"Parsing: {file_path}...")
        parsed_pages = parser.load_data(file_path)
        
        for page in parsed_pages:
            doc = Document(
                page_content=page.text,
                metadata={
                    "source": file_path,
                    "file_name": os.path.basename(file_path),
                    "type": "supply_chain_report"
                }
            )
            langchain_documents.append(doc)
    except Exception as e:
        print(f"Error parsing {os.path.basename(file_path)}: {e}")

print(f"\nSuccessfully processed all PDFs. Total chunks: {len(langchain_documents)}")

Found 3 PDF files to process.
Parsing: C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs\Economic_Security_in_a_Changing_World.pdf...
Started parsing the file under job_id 16d8e647-58d8-47a7-aa9f-2648b828b865
Parsing: C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs\thomson-reuters-founders-share-company-limited-1218.pdf...
Error while parsing the file 'C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs\thomson-reuters-founders-share-company-limited-1218.pdf': Event loop is closed
Parsing: C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs\vulnerabilities_in_the_semiconductor_supply_chain.pdf...
Started parsing the file under job_id cfd70ef2-b847-4439-9e98-1c2fd53a12e9

Successfully processed all PDFs. Total chunks: 138


In [65]:
print(local_documents[20].page_content)

   19 
ECONOMIC SECURITY IN A CHANGING WORLD © OECD 2025 
  
Defining the extent to which countries rely on significantly fewer suppliers (national import concentration) 
than is offered by the global economy (global export concentration) as significant import concentration 
reveals that the overall incidence of such significant concentration of national imports has been on the rise 
in the investigated period. This has been mainly accounted for by non-OECD economies as significant 
import concentration has not changed much on average for OECD countries (Figure 1.3, Panel A). This 
suggests that to some extent firms and consumers in OECD countries have been able to take advantage 
of diversification possibilities offered by international markets to diversify and reduce dependency.  
Figure 1.3. Average incidence of significant import concentration by country grouping 
A. Average number of imported HS6 products with significant import concentration per country 
 
B. Number of imported 

### LlamaParse and other pdf loaders are not working great on images and chartr/plots

### Use Open AI vision model to load pdf files. It will vision all pages and write texts as it is and image/charts also in a nice manner

In [77]:
import os
import glob
import base64
import fitz  # PyMuPDF is only used here as a high-speed renderer
from openai import OpenAI
from langchain_core.documents import Document

# 1. Setup paths
pdf_dir = "C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs/"
pdf_files = glob.glob(os.path.join(pdf_dir, "*.pdf"))

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
final_rag_documents = []

# 2. Iterate through reports
for file_path in pdf_files:
    file_name = os.path.basename(file_path)
    if os.path.isdir(file_path): 
        continue
        
    print(f"\n[PROCESSING VIA VISION]: {file_name}")
    doc = fitz.open(file_path)
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        
        # Step A: Take a crisp, high-resolution screenshot of the whole page (300 DPI)
        # This converts vector charts and layout text into a single unified image matrix
        pix = page.get_pixmap(dpi=300)
        img_bytes = pix.tobytes("png")
        base64_image = base64.b64encode(img_bytes).decode('utf-8')
        
        print(f"--> Transmitting Page {page_num + 1} layout to Vision Agent...")
        
        # Step B: Let the Vision Model extract the clean markdown layout
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text", 
                                "text": (
                                    "You are a strict data-extraction engine. Convert this document page layout into Markdown.\n"
                                    "1. Extract all paragraphs exactly as written.\n"
                                    "2. If a data table is visible, reconstruct it perfectly as a Markdown table.\n"
                                    "3. If a visual line graph, chart, or infographic is present, transcribe its title, labels, "
                                    "axes, and provide a comprehensive, highly accurate paragraph describing its specific data trends. "
                                    "Do NOT leave out text or clump numbers together."
                                )
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/png;base64,{base64_image}"
                                }
                            }
                        ]
                    }
                ],
                max_tokens=1500
            )
            
            structured_markdown = response.choices[0].message.content
            
            # Step C: Append the output to your RAG LangChain document array
            langchain_doc = Document(
                page_content=structured_markdown,
                metadata={
                    "source": file_path,
                    "file_name": file_name,
                    "page_number": page_num + 1,
                    "type": "supply_chain_report"
                }
            )
            final_rag_documents.append(langchain_doc)
            
        except Exception as e:
            print(f"    Error processing page {page_num + 1}: {e}")
            
    doc.close()

print(f"\n[SUCCESS] Ingested {len(final_rag_documents)} multimodal pages safely into your database context.")


[PROCESSING VIA VISION]: Economic_Security_in_a_Changing_World.pdf
--> Transmitting Page 1 layout to Vision Agent...
--> Transmitting Page 2 layout to Vision Agent...
--> Transmitting Page 3 layout to Vision Agent...
--> Transmitting Page 4 layout to Vision Agent...
--> Transmitting Page 5 layout to Vision Agent...
--> Transmitting Page 6 layout to Vision Agent...
--> Transmitting Page 7 layout to Vision Agent...
--> Transmitting Page 8 layout to Vision Agent...
--> Transmitting Page 9 layout to Vision Agent...
--> Transmitting Page 10 layout to Vision Agent...
--> Transmitting Page 11 layout to Vision Agent...
--> Transmitting Page 12 layout to Vision Agent...
--> Transmitting Page 13 layout to Vision Agent...
--> Transmitting Page 14 layout to Vision Agent...
--> Transmitting Page 15 layout to Vision Agent...
--> Transmitting Page 16 layout to Vision Agent...
--> Transmitting Page 17 layout to Vision Agent...
--> Transmitting Page 18 layout to Vision Agent...
--> Transmitting Page 1

In [6]:
len(final_rag_documents)

138

In [7]:
print(final_rag_documents[20].page_content)

Defining the extent to which countries rely on significantly fewer suppliers (national import concentration) than is offered by the global economy (global export concentration) as *significant import concentration* reveals that the overall incidence of such significant concentration of national imports has been on the rise in the investigated period. This has been mainly accounted for by non-OECD economies as significant import concentration has not changed much on average for OECD countries (Figure 1.3, Panel A). This suggests that to some extent firms and consumers in OECD countries have been able to take advantage of diversification possibilities offered by international markets to diversify and reduce dependency.

### Figure 1.3. Average incidence of significant import concentration by country grouping

#### A. Average number of imported HS6 products with significant import concentration per country

![Graph A](embedded-image)

- **Axes**: 
  - **X-axis**: Years (1997-99, 2002-04, 

### Open AI loader is working great. Lets save the file as generating again will cost ~2 USD

In [ ]:
# import pickle
# import os

# # 1. Define the save path in your project directory
# save_dir = "C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs/"
# save_path = os.path.join(save_dir, "vision_processed_docs.pkl")

# # 2. Save the list to disk
# with open(save_path, "wb") as f:
#     pickle.dump(final_rag_documents, f)

# print(f"[SAVED SUCCESS] Successfully backed up {len(final_rag_documents)} documents to:\n{save_path}")

[SAVED SUCCESS] Successfully backed up 138 documents to:
C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs/vision_processed_docs.pkl


In [5]:
#load final RAG documents from local drive
import pickle
import os

# 1. Define the path where the file is stored
#save_dir = "C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs/"
#save_path = os.path.join(save_dir, "vision_processed_docs.pkl")
save_path = "../data/processed/vision_processed_docs.pkl"

# 2. Check if the file exists before attempting to load it
if os.path.exists(save_path):
    with open(save_path, "rb") as f:  # "rb" stands for read-binary
        final_rag_documents = pickle.load(f)
        
    print(f"[LOAD SUCCESS] Successfully loaded {len(final_rag_documents)} documents from:\n{save_path}")
else:
    print(f"[ERROR] File not found at: {save_path}\nPlease check the path and try again.")

[LOAD SUCCESS] Successfully loaded 138 documents from:
C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/pdfs/vision_processed_docs.pkl


### Load word files too and merge them in pdf's final file

In [8]:
import os
import glob
from docx import Document as DocxReader
from langchain_core.documents import Document

# 1. Updated path to look outside 'util' and into your raw data directory
pdf_dir = "../data/raw/RAG_data/"
word_files = glob.glob(os.path.join(pdf_dir, "*.docx"))

word_documents = []
print(f"Found {len(word_files)} Word files to process.")

for file_path in word_files:
    # Normalize the path format so it works seamlessly on Windows, Linux, or Codespaces
    file_path = file_path.replace("\\", "/")
    file_name = os.path.basename(file_path)
    print(f"--> Reading text from Word document: {file_name}")
    
    try:
        # Open and read the document paragraph by paragraph
        doc_obj = DocxReader(file_path)
        full_text = []
        
        for paragraph in doc_obj.paragraphs:
            if paragraph.text.strip():
                full_text.append(paragraph.text.strip())
                
        # Join paragraphs into a clean cohesive text stream
        combined_text = "\n\n".join(full_text)
        
        # Structure it as a standard LangChain Document using the new repository path
        langchain_doc = Document(
            page_content=combined_text,
            metadata={
                "source": file_path,  # Now dynamically saves as a clean relative repo path
                "file_name": file_name,
                "page_number": 1,
                "type": "supply_chain_report"
            }
        )
        word_documents.append(langchain_doc)
        
    except Exception as e:
        print(f"    Failed to parse {file_name}: {e}")

print(f"\n[SUCCESS] Extracted {len(word_documents)} Word documents separately.")

Found 3 Word files to process.
--> Reading text from Word document: Disruptions at Red Sea route.docx
--> Reading text from Word document: Foxconn.docx
--> Reading text from Word document: thomson-reuters-founders-share-company-limited-1218.docx

[SUCCESS] Extracted 3 Word documents separately.


In [9]:
# Merge it with the existing RAG documents
final_rag_documents.extend(word_documents)

In [10]:
print(final_rag_documents[140].page_content[:100])

Thomson Reuters Founders Share Company Limited

Trust

Integrity

Independence

Freedom from bias

F


## Chunking

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize the recursive splitter with semantic separators
# It splits by paragraphs (\n\n), then lines (\n), then spaces ( ), as needed.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

print(f"Splitting {len(final_rag_documents)} source documents...")

# 2. Slice the complete collection into chunks
final_chunks = text_splitter.split_documents(final_rag_documents)

print(f"\n[SUCCESS] Document splitting complete!")
print(f"Total source documents processed: {len(final_rag_documents)}")
print(f"Total vector database chunks created: {len(final_chunks)}")

Splitting 141 source documents...

[SUCCESS] Document splitting complete!
Total source documents processed: 141
Total vector database chunks created: 698


In [12]:
# Print the first chunk to check the structure
print("=== FIRST CHUNK CONTENT ===")
print(final_chunks[0].page_content)
print("\n=== CHUNK METADATA ===")
print(final_chunks[0].metadata)

=== FIRST CHUNK CONTENT ===
```markdown
# New Approaches to Economic Challenges

## Economic Security in a Changing World
```

There is no visible data table to extract.

**Visual Description:**

The cover prominently features a digital lock icon overlaid on a complex background of bar graphs and line graphs, integrated into a futuristic, digital-themed design. The graphic includes several circular data indicators, displaying percentages: "42%", "97%", and "89%". These figures likely represent security or economic metrics. The data seems to suggest a visual emphasis on securing economic data or processes, underscoring the importance of economic security in a technologically advanced world. The clean, modern design indicates trends in rising economic security measures, as demonstrated by the upward trajectory of the line graph, suggesting an increase in security implementation or effectiveness over time.

=== CHUNK METADATA ===
{'source': 'C:/Users/v2garg/Desktop/IISC_Course/capstone_pr

## Create Embeddings using open AI model

In [ ]:
import os
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 1. Initialize the embedding model (uses your OPENAI_API_KEY env variable)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. Define a local folder path where Chroma will physically save the vector database files
persist_directory = "../data/processed/chroma_db"

print(f"Vectorizing {len(final_chunks)} chunks and building local database...")

# 3. Create the database and index the chunks
vector_store = Chroma.from_documents(
    documents=final_chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"\n[SUCCESS] Vector Database successfully built and saved to disk!")
print(f"Database location: {persist_directory}")

Vectorizing 698 chunks and building local database...

[SUCCESS] Vector Database successfully built and saved to disk!
Database location: C:/Users/v2garg/Desktop/IISC_Course/capstone_project/RAG Corpus/chroma_db


In [124]:
# Create a retriever object from our database
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # Pull top 3 most relevant chunks

# Run a test query about the specific charts we processed
query = "what is the flow of semiconductor's production?"
retrieved_docs = retriever.invoke(query)

print(f"=== RETRIEVAL RESULTS FOR: '{query}' ===\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('file_name')} (Page {doc.metadata.get('page_number')})")
    print(f"Snippet:\n{doc.page_content[:3000]}...\n")

=== RETRIEVAL RESULTS FOR: 'what is the flow of semiconductor's production?' ===

--- Result 1 ---
Source: vulnerabilities_in_the_semiconductor_supply_chain.pdf (Page 7)
Snippet:
Figure 1. Semiconductor production ......................................................................................................11  
Figure 2. Semiconductors are a crucial input into a range of industries ................................................15  
Figure 3. Semiconductor production has become increasingly concentrated in few Asian economies..16  
Figure 4. Semiconductor production is highly upstream and highly concentrated .................................17  
Figure 5. China, Korea and Chinese Taipei lead different segments of the semiconductors industry ....18  
Figure 6. Exports of semiconductor machinery are even more geographically concentrated than exports of semiconductors .................................................................................................................

### Create RAG pipe line for retriever

In [126]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize the central Brain (LLM)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 2. Establish your Vector Store as the official data retriever (grabs top 4 chunks)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# 3. Create a strict System Prompt ensuring answers are grounded only in your corpus
system_prompt = (
    "You are an expert global supply chain and economic intelligence analyst.\n"
    "Answer the user's question using ONLY the provided context blocks below. "
    "The context blocks include exact report paragraphs, document excerpts, "
    "and precise multi-modal vision-generated analysis summaries of charts and graphs.\n\n"
    "If you do not know the answer, or if the context does not explicitly contain "
    "the information to formulate an answer, state directly that you cannot find it in the corpus.\n\n"
    "Context:\n{context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 4. Helper function to cleanly string-format the retrieved document chunks
def format_docs(docs):
    return "\n\n".join(f"[Source: {d.metadata.get('file_name', 'Unknown')}] {d.page_content}" for d in docs)

# 5. Build the Modern LCEL RAG Chain
# This pipes your input to the retriever, formats the docs, populates the prompt, and runs the LLM
rag_pipeline = (
    {
        "context": retriever | format_docs, 
        "input": RunnablePassthrough()
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

print("[SUCCESS] Pure LCEL Modern RAG Pipeline is fully operational!")

[SUCCESS] Pure LCEL Modern RAG Pipeline is fully operational!


### Testing retriever

In [128]:
user_query = "What are the 4 main dimensions or chapters covered in the OECD report 'Economic Security in a Changing World'?"
# Run the query seamlessly using the modern pipeline
print("Analyzing corpus documents and charts...\n")
response = rag_pipeline.invoke(user_query)

print("==================== RAG ENGINE ANSWER ====================\n")
print(response)

Analyzing corpus documents and charts...

==================== RAG ENGINE ANSWER ====================

I cannot find the specific four main dimensions or chapters covered in the OECD report 'Economic Security in a Changing World' in the provided corpus.


In [129]:
user_query = "What specific company or entity is discussed regarding the document 'Foxconn.docx', and what is the main topic of that file?"
response = rag_pipeline.invoke(user_query)
print(response)

The document 'Foxconn.docx' discusses the company Foxconn, which is the world's largest iPhone factory and Apple's biggest iPhone maker. The main topic of the file is the crisis at Foxconn's plant in Zhengzhou, China, which has been disrupted by mistrust, miscommunication, and COVID-19 curbs. This crisis has led to production disruptions and affected Apple's share price. The document highlights issues such as labor unrest, communication problems, and the impact of China's strict COVID-19 policies on Foxconn's operations.


In [ ]:
user_query = "Can u briefly explain this topic and what different charts are showing? Special focus : Critical raw materials supply chains?"
response = rag_pipeline.invoke(user_query)
print(response)

The topic "Special focus: Critical raw materials supply chains" discusses the concentration and supply dynamics of critical raw materials (CRMs) globally. The production of these materials is highly concentrated in a few locations, which dominate the global supply. This concentration poses risks to supply chain resilience, prompting governments to adopt policies focused on securing CRM supplies. The report highlights the growing demand for CRMs and the need for responsible sourcing, addressing trade dependencies, and ensuring the security of supply.

Regarding the charts mentioned, they illustrate the concentration levels of raw materials over different periods. The Herfindahl-Hirschman Index (HHI) values for 2011/19 show higher concentration levels for "All raw materials" and "Critical raw materials" compared to 2007/09. Specific materials like Cobalt-NFMM and Magnesium-NFMM have shown significant increases in concentration. The charts also display shifts in export concentration, with

In [13]:
import os
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# Using your exact local absolute path
db_dir = r"C:\Users\v2garg\Desktop\IISC_Course\capstone_project\RAG Corpus\chroma_db"

print(f"Checking physical local path presence: {db_dir}")
print(f"Does folder exist locally? -> {os.path.exists(db_dir)}")

if os.path.exists(db_dir):
    print("Files present inside local chroma_db folder:", os.listdir(db_dir))
    
    try:
        # Initialize your embedding model 
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        
        db = Chroma(
            persist_directory=db_dir, 
            embedding_function=embeddings
        )
        
        vector_count = db._collection.count()
        print(f"\n[LOCAL SUCCESS] Connected to local database!")
        print(f"Total actual vectors stored on disk: {vector_count}")
        
        if vector_count > 0:
            sample_docs = db.similarity_search("disruption", k=1)
            print(f"Sample retrieval successful! Source: {sample_docs[0].metadata.get('file_name')}")
            
    except Exception as e:
        print(f"\n[ERROR] Connected to folder, but failed to extract data: {e}")
else:
    print("\n[ERROR] Path not found. Double-check the path string format.")

C:\Users\v2garg\AppData\Local\Temp\ipykernel_2592\3800327372.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Checking physical local path presence: C:\Users\v2garg\Desktop\IISC_Course\capstone_project\RAG Corpus\chroma_db
Does folder exist locally? -> True
Files present inside local chroma_db folder: ['chroma.sqlite3', 'e038ab3f-ea69-4df6-9ce0-a5f1bbf742f8']


C:\Users\v2garg\AppData\Local\Temp\ipykernel_2592\3800327372.py:18: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(



[LOCAL SUCCESS] Connected to local database!
Total actual vectors stored on disk: 698
Sample retrieval successful! Source: vulnerabilities_in_the_semiconductor_supply_chain.pdf
